In [ ]:
!pip install groq

In [ ]:
#STAGE1: Generating Relevant Intent & Corresponding Business Event Relevant to PS
STAGE1_PROMPT_TEMPLATE = """
You are designing customer-service intelligence for a contact center focused on KEY BUSINESS EVENTS.

Your goal:
Given a DOMAIN and N_QUERIES, generate a realistic set of CUSTOMER INTENTS
(why people call) and the KEY BUSINESS EVENTS that are operationally significant
and require analysis in contact centers. Focus on HIGH-IMPACT events that affect business outcomes, costs, risks, and customer experience.

DOMAIN EXAMPLES (not exhaustive)
- Consumer Finance, Lending & Banking, Healthcare Provider/Patient Services, Fintech/Payroll-as-a-Service, Insurance, Education & Student Support
- Technology/Contact Center Solutions, Healthcare Payer/Member Services, Financial Services/Debt Relief, Logistics/Delivery Services


KEY BUSINESS EVENT TAXONOMY (HIGH-IMPACT & OPERATIONALLY SIGNIFICANT)
CRITICAL BUSINESS EVENTS TO FOCUS ON:
A. CUSTOMER ESCALATION & RETENTION EVENTS:
   - Supervisor escalation, Manager escalation, Legal/compliance escalation, Churn intent signals, Service cancellation, Customer retention success, Churn prevention

B. FINANCIAL & PAYMENT EVENTS:
   - Refund requests/approvals/denials, Payment failures, Chargebacks, Fee disputes, Loan application approvals/denials, Credit limit changes

C. SERVICE DELIVERY & OPERATIONAL EVENTS:
   - Service outages, Delivery failures, Appointment no-shows, Technician dispatch failures, First-call resolution, Service quality complaints

D. COMPLIANCE & SECURITY EVENTS:
   - Fraud suspicion/confirmation, KYC verification failures, Data privacy breaches, Regulatory compliance issues, Document verification failures

E. PRODUCT & FEEDBACK EVENTS:
   - Product improvement suggestions, Feature requests, Bug reports, Customer feedback trends, Upsell/cross-sell opportunities

F. SENTIMENT & EXPERIENCE EVENTS:
   - Customer frustration escalation, Anger management incidents, Trust restoration, Satisfaction improvement, Negative sentiment loops


WHAT YOU MUST GENERATE
1. CUSTOMER INTENTS
   - These are the reasons customers call that lead to KEY BUSINESS EVENTS.
   - Focus on intents that typically result in operationally significant outcomes.
   - For diversity, scale number of intents based on N_QUERIES:
       - If N_QUERIES ≤ 6 → generate 3–5 intents
       - If 7–15 → generate 6–10 intents
       - If >15 → generate 10–18 intents
   - Create domain-specific intents that align with KEY BUSINESS EVENTS

2. KEY BUSINESS EVENTS - OPERATIONALLY SIGNIFICANT:
   - These must be HIGH-IMPACT events that businesses need to analyze causally.
   - Each intent should produce 1–3 KEY business events from the taxonomy above.
   - Focus on events mentioned in the problem statement: escalations, refunds, churn, product improvements.
   - Events should be measurable, trackable, and operationally relevant.

3. REALISM CONSTRAINTS
   - Generate only KEY BUSINESS EVENTS that would trigger analytical interest.
   - Events should be specific enough for causal analysis.
   - Avoid trivial or routine events that don't require root-cause investigation.
   - Ensure events are domain-appropriate and operationally significant.


OUTPUT FORMAT (STRICT JSON)
Return a single JSON object:
{{
  "domain": "{domain}",
  "n_queries": {n_queries},
  "intents": [
    {{
      "intent": "<customer intent leading to key business events>",
      "short_desc": "<1-line why customers call>",
      "business_events": [
        "<KEY business event 1 - operationally significant>",
        "<KEY business event 2 - operationally significant>",
        "<KEY business event 3 - operationally significant>"
      ]
    }}
  ]
}}

RULES:
- Pure JSON only.
- No markdown.
- No commentary outside JSON.
- FOCUS ON KEY BUSINESS EVENTS: escalations, refunds, churn, product improvements, and other operationally significant outcomes.
"""

In [ ]:
# ========== STAGE 1 EVALUATION PROMPT ==========
STAGE1_JUDGE_PROMPT = """
You are an evaluator for contact-center analytics.

Your job is to score the quality of Stage-1 JSON output that contains:
- customer intents
- associated KEY BUSINESS EVENTS

You must validate whether:
1. Intents are domain-appropriate and specific.
2. Each intent leads to operationally significant *business events*.
3. Business events belong to HIGH-IMPACT categories only:
   - escalations
   - refunds or payment failures
   - denials/approvals
   - churn/cancellation
   - fraud/KYC/security
   - service outages or failures
   - complaints or disputes
4. Events are measurable, trackable, and operational.
5. Avoid trivial events: "information provided", "general inquiry", "customer had a question".

OUTPUT A STRICT JSON OBJECT:

{
  "quality_score": "<integer 1-10>",
  "low_quality_reasons": ["reason 1", "reason 2", ...],
  "should_retry": "<yes | no>",
  "detailed_feedback": "<comprehensive feedback on what works well and what needs improvement>",
  "strengths": ["strength 1", "strength 2", ...],
  "improvement_suggestions": ["suggestion 1", "suggestion 2", ...]
}

Scoring Rubric:
10-9: Excellent - All events are high-impact, domain-specific, and operationally significant
8-7: Good - Mostly high-impact events with minor issues
6-5: Fair - Mixed quality with some trivial events
4-1: Poor - Mostly trivial events or poor domain alignment

Rules:
- If score ≤ 6 → should_retry = "yes"
- If more than 35% events are trivial → should_retry = "yes"
- If JSON structure is invalid → should_retry = "yes"

Now evaluate the following Stage-1 output:

STAGE1_CONTENT:
```json
{stage1_json}
Provide a thorough evaluation focusing on business impact and operational significance.
"""

In [ ]:
import os
import json
import time
import random
import re
from typing import Dict, Any, List, Optional
from tqdm import tqdm

# CONFIGURATION
GROQ_API_KEY = "gsk_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"
GENERATION_MODEL = "openai/gpt-oss-20b"  # For Stage 1 generation
EVALUATION_MODEL = "llama-3.1-8b-instant"  # For evaluation
MAX_RETRIES = 5
BASE_BACKOFF = 1.0
OUTPUT_DIR = "stage1_evaluation_outputs"
EVALUATION_HISTORY_FILE = "evaluation_history.json"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# MECE FRAMEWORK
MECE_CATEGORIES = ["Diagnostic", "Descriptive", "Comparative", "Contextual"]

SUBCATEGORY_OPTIONS = {
    "Diagnostic": [
        "Causal Root-Analysis", "Behavioral Cause Analysis", "Emotional & Linguistic Causality",
        "Process / Policy Cause Analysis", "ASR / Misunderstanding Causality",
        "Temporal Causality & Early Signals", "Counterfactual Causality"
    ],
    "Descriptive": [
        "Pattern Identification", "Correlation Analysis", "Statistical Summarization",
        "Temporal Trend Analysis", "Segmented Behavior Analysis", "Event-Type Pattern Discovery"
    ],
    "Comparative": [
        "Event-Type Comparison", "Agent Performance Comparison", "Customer Segment Comparison",
        "Process / Policy Comparison", "Cross-Domain Comparison", "Scenario-Based Contrastive Analysis"
    ],
    "Contextual": [
        "Evidence Retrieval", "Contextual Follow-Up", "Clarification / Drill-Down",
        "Refinement Queries", "Counterfactual / Exception Follow-Up", "Multi-Turn Analytical Query Chains",
        "Cross-Case Evidence Linking"
    ]
}

# JUDGE PROMPT FOR UPLOADED SCHEMA
UPLOADED_SCHEMA_JUDGE_PROMPT = """You are a quality assurance judge for evaluating uploaded Stage 1 schema. Your task is to evaluate the quality of the provided schema containing business events and intents.

UPLOADED SCHEMA TO EVALUATE:
{uploaded_schema}

EVALUATION CRITERIA:
1. **Schema Structure**: Does it follow the proper schema format with metadata, stage1_business_events, and stage1_intents?
2. **Business Events Quality**: Are business events specific, actionable, and relevant to the domain?
3. **Intents Quality**: Are intents well-defined with proper short descriptions?
4. **Mapping Quality**: Are business events properly mapped to intents in the stage1_intents section?
5. **Diversity**: Is there good variety in both business events and intents?
6. **Relevance**: Are all elements relevant to the specified domain?
7. **Completeness**: Are all required fields present and properly populated?

SCORING GUIDE:
- 9-10: Excellent - Well-structured, relevant, diverse, proper mappings. ACCEPT.
- 7-8: Good - Minor issues but generally good quality. ACCEPT.
- 5-6: Fair - Noticeable issues but usable. ACCEPT WITH NOTES.
- 1-4: Poor - Major structural or relevance problems. REJECT.

Respond with JSON in this exact format:
{{
    "quality_score": 8,
    "evaluation_status": "accepted",
    "reasoning": "Brief explanation of your evaluation",
    "strengths": ["List of strengths"],
    "improvement_areas": ["List of areas needing improvement"],
    "schema_issues": ["List any schema structure issues found"],
    "mapping_issues": ["List any intent-business event mapping issues"]
}}"""


# GROQ CLIENT
from groq import Groq

def get_client():
    return Groq(api_key=GROQ_API_KEY)

def backoff_sleep(attempt: int):
    time.sleep(BASE_BACKOFF * (2 ** attempt) + random.random() * 0.2)

def call_groq_chat(client: Groq, prompt: str, model: str = GENERATION_MODEL) -> str:
    """Call Groq API with retry logic"""
    last_err = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,
                response_format={"type": "json_object"}
            )
            content = resp.choices[0].message.content
            if content is None:
                raise ValueError("Empty response from LLM")
            return content.strip()
        except Exception as e:
            last_err = e
            if attempt == MAX_RETRIES - 1:
                raise
            backoff_sleep(attempt)
    raise RuntimeError(f"Groq failed after retries: {last_err}")

def extract_first_json_blob(text: str) -> Any:
    """Robust JSON extraction from LLM response"""
    if not isinstance(text, str):
        raise ValueError("Assistant returned non-string content")

    text = re.sub(r"```(json)?", "", text).strip()

    # Try direct parse first
    try:
        return json.loads(text)
    except:
        pass

    # Find JSON object
    obj_match = re.search(r"\{(?:[^{}]|(?R))*\}", text, re.DOTALL)
    if obj_match:
        candidate = obj_match.group(0)
        # Clean common JSON issues
        candidate = candidate.replace("\r", "").replace("'", '"')
        candidate = re.sub(r",\s*([}\]])", r"\1", candidate)
        try:
            return json.loads(candidate)
        except:
            pass

    raise ValueError("No valid JSON found in response")

def safe_json_parse(text: str, default: Dict = None) -> Dict:
    """Safely parse JSON with fallback"""
    try:
        return json.loads(text)
    except:
        return default or {"error": "Failed to parse JSON"}

def judge_stage1_output(stage1_json: Dict[str, Any]) -> Dict[str, Any]:
    """Evaluate Stage 1 output quality using separate evaluation model"""
    client = get_client()

    prompt = STAGE1_JUDGE_PROMPT.replace("{stage1_json}", json.dumps(stage1_json, indent=2))

    try:
        resp = client.chat.completions.create(
            model=EVALUATION_MODEL,  # Different model for evaluation
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            response_format={"type": "json_object"}
        )

        judged = resp.choices[0].message.content.strip()
        judge_result = safe_json_parse(judged)

        # Ensure all required fields exist with safe defaults
        required_fields = {
            "quality_score": 5,
            "should_retry": "yes",
            "reasoning": "Default reasoning - JSON parsing issue",
            "strengths": ["No strengths identified"],
            "improvement_areas": ["No specific areas identified"]
        }

        for field, default in required_fields.items():
            if field not in judge_result:
                judge_result[field] = default

        return judge_result

    except Exception as e:
        print(f"Judge model error: {e}")
        # Return default evaluation result if judge fails
        return {
            "quality_score": 3,
            "should_retry": "yes",
            "reasoning": f"Judge model failed: {str(e)}",
            "strengths": ["None - evaluation failed"],
            "improvement_areas": ["Judge model connectivity", "Response parsing"]
        }

def judge_uploaded_schema(uploaded_schema: Dict[str, Any]) -> Dict[str, Any]:
    """Evaluate uploaded schema containing Stage 1 output"""
    client = get_client()

    prompt = UPLOADED_SCHEMA_JUDGE_PROMPT.replace("{uploaded_schema}", json.dumps(uploaded_schema, indent=2))

    try:
        resp = client.chat.completions.create(
            model=EVALUATION_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            response_format={"type": "json_object"}
        )

        judged = resp.choices[0].message.content.strip()
        judge_result = safe_json_parse(judged)

        # Ensure all required fields exist with safe defaults
        required_fields = {
            "quality_score": 5,
            "evaluation_status": "accepted_with_issues",
            "reasoning": "Default reasoning - JSON parsing issue",
            "strengths": ["No strengths identified"],
            "improvement_areas": ["No specific areas identified"],
            "schema_issues": ["No schema issues identified"],
            "mapping_issues": ["No mapping issues identified"]
        }

        for field, default in required_fields.items():
            if field not in judge_result:
                judge_result[field] = default

        return judge_result

    except Exception as e:
        print(f"Judge model error: {e}")
        return {
            "quality_score": 3,
            "evaluation_status": "rejected",
            "reasoning": f"Judge model failed: {str(e)}",
            "strengths": ["None - evaluation failed"],
            "improvement_areas": ["Judge model connectivity", "Response parsing"],
            "schema_issues": ["Evaluation system error"],
            "mapping_issues": ["Evaluation system error"]
        }

def run_stage1(domain: str, n_queries: int) -> Dict[str, Any]:
    """Run Stage 1 with quality evaluation using separate models"""
    client = get_client()

    for attempt in range(MAX_RETRIES):
        print(f"Stage 1 Generation Attempt {attempt+1}/{MAX_RETRIES}")

        # Generate Stage 1 output using generation model
        prompt = STAGE1_PROMPT_TEMPLATE.format(domain=domain, n_queries=n_queries)
        raw = call_groq_chat(client, prompt, model=GENERATION_MODEL)
        parsed = extract_first_json_blob(raw)

        # Judge the output using evaluation model
        judge_result = judge_stage1_output(parsed)

        print(f"   Judge Score: {judge_result['quality_score']}/10")
        print(f"   Should Retry: {judge_result['should_retry']}")
        print(f"   Reasoning: {judge_result.get('reasoning', 'No reasoning provided')}")

        if judge_result["should_retry"].lower() == "no":
            print("Stage 1 accepted.")
            return {
                "stage1_output": parsed,
                "judgment": judge_result,
                "attempts": attempt + 1,
                "status": "accepted"
            }

        print("Judge rejected output → retrying Stage 1...")
        backoff_sleep(attempt)

    print("Final fallback: Returning last Stage-1 output despite quality issues.")
    return {
        "stage1_output": parsed,
        "judgment": judge_result,
        "attempts": MAX_RETRIES,
        "status": "rejected_but_used"
    }

def extract_business_events(stage1_output: Dict[str, Any]) -> list:
    """Extract unique business events from Stage 1 output"""
    business_events = []

    # Handle different schema formats
    if "business_events" in stage1_output:
        events = stage1_output["business_events"]
        if isinstance(events, list):
            business_events.extend(events)

    # Also extract from stage1_business_events (uploaded schema format)
    if "stage1_business_events" in stage1_output:
        events = stage1_output["stage1_business_events"]
        if isinstance(events, list):
            business_events.extend(events)

    # Also extract from intents structure
    if "intents" in stage1_output:
        for intent in stage1_output["intents"]:
            if isinstance(intent, dict) and "business_events" in intent:
                business_events.extend(intent["business_events"])

    # Also extract from stage1_intents (uploaded schema format)
    if "stage1_intents" in stage1_output:
        for intent in stage1_output["stage1_intents"]:
            if isinstance(intent, dict) and "business_events" in intent:
                business_events.extend(intent["business_events"])

    # Remove duplicates while preserving order
    seen = set()
    unique_events = []
    for event in business_events:
        if event and str(event) not in seen:
            seen.add(str(event))
            unique_events.append(event)

    return unique_events

def extract_intents(stage1_output: Dict[str, Any]) -> list:
    """Extract intents from Stage 1 output"""
    intents = []

    # Handle different schema formats
    if "intents" in stage1_output:
        intents_data = stage1_output["intents"]
        if isinstance(intents_data, list):
            intents.extend(intents_data)

    if "stage1_intents" in stage1_output:
        intents_data = stage1_output["stage1_intents"]
        if isinstance(intents_data, list):
            intents.extend(intents_data)

    return intents

def load_json_file(file_path: str) -> Dict[str, Any]:
    """Load and parse JSON file"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception as e:
        raise ValueError(f"Failed to load JSON file: {e}")

def load_evaluation_history() -> List[Dict]:
    """Load previous evaluation results"""
    try:
        if os.path.exists(EVALUATION_HISTORY_FILE):
            with open(EVALUATION_HISTORY_FILE, 'r', encoding='utf-8') as f:
                return json.load(f)
    except Exception as e:
        print(f"Warning: Could not load evaluation history: {e}")
    return []

def save_evaluation_history(history: List[Dict]):
    """Save evaluation results to history file"""
    try:
        with open(EVALUATION_HISTORY_FILE, 'w', encoding='utf-8') as f:
            json.dump(history, f, indent=2, ensure_ascii=False)
    except Exception as e:
        print(f"Warning: Could not save evaluation history: {e}")

def add_to_evaluation_history(evaluation_result: Dict):
    """Add a new evaluation result to history"""
    history = load_evaluation_history()

    # Add timestamp if not present
    if "timestamp" not in evaluation_result:
        evaluation_result["timestamp"] = time.strftime("%Y-%m-%d %H:%M:%S")

    history.append(evaluation_result)

    # Keep only last 100 evaluations to prevent file from growing too large
    if len(history) > 100:
        history = history[-100:]

    save_evaluation_history(history)

def get_evaluation_stats() -> Dict:
    """Get statistics from evaluation history"""
    history = load_evaluation_history()

    if not history:
        return {"total_evaluations": 0, "average_score": 0}

    scores = [eval.get("evaluation", {}).get("quality_score", 0) for eval in history]
    valid_scores = [score for score in scores if isinstance(score, (int, float))]

    stats = {
        "total_evaluations": len(history),
        "average_score": sum(valid_scores) / len(valid_scores) if valid_scores else 0,
        "recent_evaluations": history[-10:]  # Last 10 evaluations
    }

    return stats

# ========== MAIN EVALUATION FUNCTIONS ==========
def evaluate_stage1_output(
    domain: str,
    n_events: int = 20,
    save_output: bool = True,
    store_in_history: bool = True
) -> Dict[str, Any]:
    """
    Evaluate Stage 1 by generating new output for a domain

    Args:
        domain (str): The business domain
        n_events (int): Number of business events to generate
        save_output (bool): Whether to save results to file
        store_in_history: Whether to store in evaluation history

    Returns:
        Dict with Stage 1 results and evaluation
    """

    print(f"Starting Stage 1 evaluation...")
    print(f"   Domain: {domain}")
    print(f"   Events: {n_events}")
    print(f"   Generation Model: {GENERATION_MODEL}")
    print(f"   Evaluation Model: {EVALUATION_MODEL}")

    # Run Stage 1 with evaluation
    print(f"\n STAGE 1: Generating and evaluating business events...")
    stage1_result = run_stage1(domain, n_events)

    # Extract business events for reporting
    business_events = extract_business_events(stage1_result["stage1_output"])
    intents = extract_intents(stage1_result["stage1_output"])

    # Prepare results
    results = {
        "metadata": {
            "domain": domain,
            "n_events": n_events,
            "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "evaluation_status": stage1_result["status"],
            "attempts_needed": stage1_result["attempts"],
            "generation_model": GENERATION_MODEL,
            "evaluation_model": EVALUATION_MODEL,
            "evaluation_type": "domain_generation"
        },
        "evaluation": stage1_result["judgment"],
        "stage1_output": stage1_result["stage1_output"],
        "business_events_summary": {
            "total_unique_events": len(business_events),
            "events": business_events
        },
        "intents_summary": {
            "total_intents": len(intents),
            "intents": intents
        }
    }

    # Store in evaluation history
    if store_in_history:
        add_to_evaluation_history(results)

    # Save results if requested
    if save_output:
        safe_domain = re.sub(r'[^A-Za-z0-9_-]+', '_', domain)
        filename = f"stage1_evaluation_{safe_domain}_{n_events}_{int(time.time())}.json"
        filepath = os.path.join(OUTPUT_DIR, filename)

        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

        print(f"Results saved to: {filepath}")

    # Print summary
    print(f"\n EVALUATION COMPLETE!")
    print(f"Summary:")
    print(f"   - Status: {stage1_result['status']}")
    print(f"   - Quality Score: {stage1_result['judgment']['quality_score']}/10")
    print(f"   - Attempts Needed: {stage1_result['attempts']}")
    print(f"   - Unique Business Events: {len(business_events)}")
    print(f"   - Total Intents: {len(intents)}")

    print(f"\nStrengths:")
    for strength in stage1_result['judgment'].get('strengths', []):
        print(f"   {strength}")

    print(f"\nImprovement Areas:")
    for area in stage1_result['judgment'].get('improvement_areas', []):
        print(f"   {area}")

    return results

def evaluate_uploaded_schema(
    file_path: str,
    save_output: bool = True,
    store_in_history: bool = True
) -> Dict[str, Any]:
    """
    Evaluate Stage 1 by uploading an existing schema file

    Args:
        file_path (str): Path to the JSON file containing Stage 1 schema
        save_output (bool): Whether to save results to file
        store_in_history: Whether to store in evaluation history

    Returns:
        Dict with evaluation results
    """

    print(f"Starting uploaded schema evaluation...")
    print(f"   File: {file_path}")
    print(f"   Evaluation Model: {EVALUATION_MODEL}")

    # Load the uploaded schema
    try:
        uploaded_schema = load_json_file(file_path)
        print(f"   Successfully loaded schema")
    except Exception as e:
        print(f"   ERROR: Failed to load schema: {e}")
        return {"error": f"Failed to load schema: {e}"}

    # Evaluate the schema
    print(f"\n EVALUATING UPLOADED SCHEMA...")
    evaluation_result = judge_uploaded_schema(uploaded_schema)

    # Extract business events and intents for reporting
    business_events = extract_business_events(uploaded_schema)
    intents = extract_intents(uploaded_schema)

    # Get domain from metadata or schema
    domain = "Unknown"
    if "metadata" in uploaded_schema and "domain" in uploaded_schema["metadata"]:
        domain = uploaded_schema["metadata"]["domain"]
    elif "domain" in uploaded_schema:
        domain = uploaded_schema["domain"]

    # Prepare results
    results = {
        "metadata": {
            "domain": domain,
            "source_file": file_path,
            "evaluated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "evaluation_status": evaluation_result["evaluation_status"],
            "evaluation_model": EVALUATION_MODEL,
            "evaluation_type": "schema_upload"
        },
        "evaluation": evaluation_result,
        "uploaded_schema": uploaded_schema,
        "business_events_summary": {
            "total_unique_events": len(business_events),
            "events": business_events
        },
        "intents_summary": {
            "total_intents": len(intents),
            "intents": intents
        }
    }

    # Store in evaluation history
    if store_in_history:
        add_to_evaluation_history(results)

    # Save results if requested
    if save_output:
        safe_domain = re.sub(r'[^A-Za-z0-9_-]+', '_', domain)
        safe_filename = os.path.splitext(os.path.basename(file_path))[0]
        filename = f"schema_evaluation_{safe_domain}_{safe_filename}_{int(time.time())}.json"
        filepath = os.path.join(OUTPUT_DIR, filename)

        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

        print(f"Results saved to: {filepath}")

    # Print summary
    print(f"\n SCHEMA EVALUATION COMPLETE!")
    print(f"Summary:")
    print(f"   - Domain: {domain}")
    print(f"   - Status: {evaluation_result['evaluation_status']}")
    print(f"   - Quality Score: {evaluation_result['quality_score']}/10")
    print(f"   - Unique Business Events: {len(business_events)}")
    print(f"   - Total Intents: {len(intents)}")

    print(f"\nStrengths:")
    for strength in evaluation_result.get('strengths', []):
        print(f"   {strength}")

    print(f"\nImprovement Areas:")
    for area in evaluation_result.get('improvement_areas', []):
        print(f"   {area}")

    print(f"\nSchema Issues:")
    for issue in evaluation_result.get('schema_issues', []):
        print(f"   {issue}")

    print(f"\nMapping Issues:")
    for issue in evaluation_result.get('mapping_issues', []):
        print(f"   {issue}")

    return results

# UNIFIED EVALUATION FUNCTION
def evaluate_stage1(
    domain: Optional[str] = None,
    file_path: Optional[str] = None,
    n_events: int = 20,
    save_output: bool = True,
    store_in_history: bool = True
) -> Dict[str, Any]:
    """
    UNIFIED FUNCTION: Evaluate Stage 1 either by domain or uploaded file

    Args:
        domain (str, optional): Domain to generate and evaluate
        file_path (str, optional): Path to JSON file to evaluate
        n_events (int): Number of events if generating
        save_output (bool): Whether to save results
        store_in_history: Whether to store in history

    Returns:
        Dict with evaluation results
    """

    if file_path:
        return evaluate_uploaded_schema(
            file_path=file_path,
            save_output=save_output,
            store_in_history=store_in_history
        )
    elif domain:
        return evaluate_stage1_output(
            domain=domain,
            n_events=n_events,
            save_output=save_output,
            store_in_history=store_in_history
        )
    else:
        raise ValueError("Either domain or file_path must be provided")

# USAGE EXAMPLES
if __name__ == "__main__":
    print("=== STAGE 1 EVALUATION SYSTEM ===")
    print(f"Configuration:")
    print(f"   - Generation Model: {GENERATION_MODEL}")
    print(f"   - Evaluation Model: {EVALUATION_MODEL}")
    print(f"   - Output Directory: {OUTPUT_DIR}")
    print(f"   - History File: {EVALUATION_HISTORY_FILE}")

    # Show existing evaluation stats
    history_stats = get_evaluation_stats()
    print(f"\nExisting Evaluation History:")
    print(f"   - Total evaluations: {history_stats['total_evaluations']}")
    print(f"   - Average score: {history_stats['average_score']:.2f}/10")

    # Example 1: Evaluate by domain (generate new)
    print("\n1. EVALUATE BY DOMAIN (Generate New):")
    results1 = evaluate_stage1(
        domain="Healthcare Provider / Patient Services",
        n_events=15
    )

    # Example 2: Evaluate uploaded schema file
    print("\n2. EVALUATE UPLOADED SCHEMA:")
    # This would be used when you have an existing JSON file
    results2 = evaluate_stage1(file_path="./queries_Telecom_HARD_20.json")

    # Example 3: Show how to use both methods
    print("\n3. USAGE EXAMPLES:")
    print("   # Evaluate by domain:")
    print("   results = evaluate_stage1(domain='E-commerce', n_events=10)")
    print("")
    print("   # Evaluate uploaded file:")
    print("   results = evaluate_stage1(file_path='queries_Telecom_HARD_20.json')")
    print("")
    print("   # Both methods store results in evaluation history")

    print("\n=== EVALUATION SYSTEM READY ===")